# **YOLO V8 Model Training**




# **Step 1 : Dual GPU Verification**

In [1]:
!nvidia-smi 

Sat Mar  7 14:40:33 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.105.08             Driver Version: 580.105.08     CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   37C    P8              9W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

# **Step 2 : Dependencies Installation**

In [2]:
%pip install ultralytics torch torchvision torchaudio --extra-index-url https://download.pytorch.org/whl/cu118

Looking in indexes: https://pypi.org/simple, https://download.pytorch.org/whl/cu118
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 20.7 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.


# **Step 3 : Dataset Download**

# **Step  4 : Model Configuration & Training**

In [4]:
from ultralytics import YOLO
import os
import shutil
import torch

# Update Data.yaml path
data_yaml_path = '/kaggle/working/mydata-9/data.yaml'

# Define the dataset directory path
dataset_base_path = "/kaggle/working/mydata-9"

#cache_dir 
cache_dir = "/kaggle/working/cache"
os.makedirs(cache_dir, exist_ok=True)

# Writable folder for YOLO cache
os.makedirs("/kaggle/working/cache", exist_ok=True)

# Writable folder for PyTorch CUDA kernels
os.makedirs("/kaggle/working/torch_cache", exist_ok=True)
os.environ["TORCH_KERNEL_CACHE_PATH"] = "/kaggle/working/torch_cache"

model = YOLO('yolov8m.pt')
# Define training parameters
train_args = {
    'data': data_yaml_path, # Keep data.yaml path
    'model': 'yolov8m.pt', # Specify yolov8m model
    'task':'detect',
    'epochs': 1, # No. of Epochs
    'batch': 16, # Batch size adjusted for T4 GPU memory
    'imgsz': 640,
    'cache': cache_dir,
    'workers': 3,
    'device': list(range(torch.cuda.device_count()))
              if torch.cuda.device_count() > 1 else 0
}

# Start training
results = model.train(**train_args)

print("Training complete.")

# Model Export 
# Paths
folder_path = "/kaggle/working/runs/detect"
zip_path = "/kaggle/working/120_cu_V3_epoch_training_output"

# Remove existing zip if it exists
if os.path.exists(zip_path + ".zip"):
    os.remove(zip_path + ".zip")

# Create zip archive
shutil.make_archive(zip_path, 'zip', folder_path)

print("ZIP file created at:", zip_path + ".zip")


Ultralytics 8.4.21 🚀 Python-3.12.12 torch-2.8.0+cu126 CUDA:0 (Tesla T4, 14913MiB)
                                                      CUDA:1 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=/kaggle/working/cache, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/mydata-9/data.yaml, degrees=0.0, deterministic=True, device=0,1, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=1, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=train, nbs=64,